# Multi-Model Assertion Dataset Comparison

This notebook compares how different language models respond to various types of assertions across different categories and dimensions. It provides comprehensive analysis of model behavior patterns, strengths, and weaknesses.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from glob import glob
import warnings
warnings.filterwarnings('ignore')
import os
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests
# Set font to match other plots

os.chdir('..')
print(os.getcwd())
sns.set_style("white")
sns.despine()
# Set up plotting style
import matplotlib
matplotlib.rcParams["mathtext.rm"] = "Bitstream Vera Sans"
matplotlib.rcParams["mathtext.it"] = "Bitstream Vera Sans:italic"
matplotlib.rcParams["mathtext.bf"] = "Bitstream Vera Sans:bold"
matplotlib.rcParams["mathtext.fontset"] = "stix"
matplotlib.rcParams["font.family"] = "STIXGeneral"
matplotlib.rcParams["font.size"] = "28"
matplotlib.rcParams["axes.spines.right"] = False
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.facecolor"] = "white"
matplotlib.rcParams["savefig.facecolor"] = "white"

# Color palette for models
MODEL_COLORS = {
    'meta_llama_Llama_3_1_8B_Instruct': '#1f77b4',
    'meta_llama_Llama_3_1_8B': '#ff7f0e',
    # Add more colors as needed for additional models
    'model3': '#2ca02c',
    'model4': '#d62728',
    'model5': '#9467bd'
}

print("Setup complete!")

def get_model_label(model_name):
    if "Llama_3_1_8B_Instruct" in model_name:
        return "LLaMA3-8B-Instruct"
    elif "Llama_3_1_8B" in model_name:
        return "LLaMA3-8B"
    elif "Llama_3_2_1B_Instruct" in model_name:
        return "LLaMA3-1B-Instruct"
    elif "Llama_3_2_3B_Instruct" in model_name:
        return "LLaMA3-3B-Instruct"
    elif "Llama_3_2_3B" in model_name:
        return "LLaMA3-3B"
    elif "Llama_3_2_1B" in model_name:
        return "LLaMA3-1B"
    elif "Qwen3_1_7B_Base" in model_name:
        return "Qwen3-1.7B-Base"
    elif "Qwen3_1_7B" in model_name:
        return "Qwen3-1.7B-Instruct"
    elif "Qwen3_8B_Base" in model_name:
        return "Qwen3-8B-Base"
    elif "Qwen3_8B" in model_name:
        return "Qwen3-8B-Instruct"
    elif "Qwen3_30B_A3B_Base" in model_name:
        return "Qwen3-30B-A3B-Base"
    elif "Qwen3_30B_A3B" in model_name:
        return "Qwen3-30B-A3B-Instruct"
    elif "gemma_3_1b_it" in model_name:
        return "Gemma3-1B-Instruct"
    elif "gemma_3_1b_pt" in model_name:
        return "Gemma3-1B"
    elif "gemma_3_12b_it" in model_name:
        return "Gemma3-12B-Instruct"
    elif "gemma_3_12b_pt" in model_name:
        return "Gemma3-12B"
    elif "gemma_3_27b_it" in model_name:
        return "Gemma3-27B-Instruct"
    elif "gemma_3_27b_pt" in model_name:
        return "Gemma3-27B"
    elif "gpt_5_nano" in model_name:
        return "GPT5-Nano"
    elif "gpt_5_mini" in model_name:
        return "GPT5-Mini"
    elif "gpt_5" in model_name:
        return "GPT5"
    else:
        raise ValueError(f"Unknown model: {model_name}")

/Users/kdu/code/rycolab/Assertions
Setup complete!


## Data Loading and Preprocessing

In [2]:
def load_model_data(data_dir='data', min_examples=1000, dataset_size="1000", query_only=False):
    """
    Load results from all available models in the data directory.
    Returns a dictionary with model names as keys and DataFrames as values.
    """
    model_data = {}
    data_path = Path(data_dir)
    
    # Find all model directories
    model_dirs = [d for d in data_path.iterdir() if d.is_dir() and not d.name.startswith('.')]
    model_dirs = [d for d in model_dirs if dataset_size in d.name] # filter in only models evaluated on 1000 examples
    
    filename = "results_query_only.csv" if query_only else "results.csv"
    generate_filename = "results_generate_query_only.csv" if query_only else "results_generate.csv"

    for model_dir in model_dirs:
        results_file = model_dir / filename
        generate_results_file = model_dir / generate_filename
        if generate_results_file.exists():
            results_file = generate_results_file
        if results_file.exists():
            model_name = model_dir.name
            print(f"Loading data for {model_name}...")
            
            try:
                df = pd.read_csv(results_file)
                df["yes_probability"] = pd.to_numeric(df["yes_probability"], errors="coerce")
                df["no_probability"] = pd.to_numeric(df["no_probability"], errors="coerce")
               
                if len(df) < min_examples:
                    print(f"Skipping {model_name} (only {len(df)} examples, below threshold of {min_examples})")
                    continue  # skip datasets below threshold
            
                df['model'] = model_name
                df['fact_id'] = df['subject'] + ' -> ' + df['object_true']
                df['confidence'] = abs(df['yes_probability'] - df['no_probability'])
                
                
                model_data[model_name] = df
                print(f"  - Loaded {len(df)} examples")
            except Exception as e:
                print(f"  - Error loading {model_name}: {e}")
        else:
            print(f"No results.csv found for {model_dir.name}")
    
    return model_data

# Load all model data
model_data = load_model_data()
print(f"\nLoaded data for {len(model_data)} models: {list(model_data.keys())}")

# Combine all data into a single DataFrame for easier analysis
if model_data:
    combined_df = pd.concat(model_data.values(), ignore_index=True)
    combined_df["query_only"] = False
    print(f"Combined dataset shape: {combined_df.shape}")
else:
    print("No model data found!")

query_only_model_data = load_model_data(query_only=True)
combined_df_query_only = pd.concat(query_only_model_data.values(), ignore_index=True)
combined_df_query_only["query_only"] = True
combined_df = pd.concat([combined_df, combined_df_query_only], ignore_index=True)
combined_df["query_only"].value_counts()


Loading data for Qwen_Qwen3_1_7B_Base_generated_assertions_v2_1000...


  - Loaded 38000 examples
Loading data for meta_llama_Llama_3_2_1B_generated_assertions_v2_1000...
  - Loaded 38000 examples
Loading data for meta_llama_Llama_3_1_8B_generated_assertions_v2_1000...


  - Loaded 38000 examples
Loading data for google_gemma_3_27b_it_generated_assertions_v2_1000...
  - Loaded 38000 examples
Loading data for google_gemma_3_1b_it_generated_assertions_v2_1000...


  - Loaded 38000 examples
Loading data for Qwen_Qwen3_8B_generated_assertions_v2_1000...
  - Loaded 38000 examples
Loading data for google_gemma_3_12b_pt_generated_assertions_v2_1000...


  - Loaded 38000 examples
Loading data for meta_llama_Llama_3_2_3B_Instruct_generated_assertions_v2_1000...


...

  - Loaded 38000 examples
Loading data for Qwen_Qwen3_30B_A3B_Base_generated_assertions_v2_1000...
  - Loaded 38000 examples
Loading data for Qwen_Qwen3_30B_A3B_generated_assertions_v2_1000...


  - Loaded 38000 examples
Loading data for meta_llama_Llama_3_1_8B_Instruct_generated_assertions_v2_1000...
  - Loaded 38000 examples
Loading data for Qwen_Qwen3_1_7B_generated_assertions_v2_1000...


  - Loaded 38000 examples
Loading data for google_gemma_3_12b_it_generated_assertions_v2_1000...
  - Loaded 38000 examples
Loading data for Qwen_Qwen3_8B_Base_generated_assertions_v2_1000...


  - Loaded 38000 examples


query_only
False    684000
True     684000
Name: count, dtype: int64

In [3]:
model_data.keys()

dict_keys(['Qwen_Qwen3_1_7B_Base_generated_assertions_v2_1000', 'meta_llama_Llama_3_2_1B_generated_assertions_v2_1000', 'meta_llama_Llama_3_1_8B_generated_assertions_v2_1000', 'google_gemma_3_27b_it_generated_assertions_v2_1000', 'google_gemma_3_1b_it_generated_assertions_v2_1000', 'Qwen_Qwen3_8B_generated_assertions_v2_1000', 'google_gemma_3_12b_pt_generated_assertions_v2_1000', 'meta_llama_Llama_3_2_3B_Instruct_generated_assertions_v2_1000', 'google_gemma_3_1b_pt_generated_assertions_v2_1000', 'google_gemma_3_27b_pt_generated_assertions_v2_1000', 'meta_llama_Llama_3_2_3B_generated_assertions_v2_1000', 'meta_llama_Llama_3_2_1B_Instruct_generated_assertions_v2_1000', 'Qwen_Qwen3_30B_A3B_Base_generated_assertions_v2_1000', 'Qwen_Qwen3_30B_A3B_generated_assertions_v2_1000', 'meta_llama_Llama_3_1_8B_Instruct_generated_assertions_v2_1000', 'Qwen_Qwen3_1_7B_generated_assertions_v2_1000', 'google_gemma_3_12b_it_generated_assertions_v2_1000', 'Qwen_Qwen3_8B_Base_generated_assertions_v2_1000']

In [4]:
# Basic overview of the dataset
if len(model_data) > 0:
    print("=== DATASET OVERVIEW ===")
    print(f"Total models: {len(model_data)}")
    print(f"Available columns: {list(combined_df.columns)}")
    
    print("\n=== MODEL STATISTICS ===")
    for model_name, df in model_data.items():
        print(f"\n{model_name}:")
        print(f"  Examples: {len(df)}")
        print(f"  Classifications: {dict(df['classification'].value_counts())}")
        print(f"  Categories: {len(df['category'].unique())} unique")
        print(f"  Dimensions: {len(df['dimension'].unique())} unique")
        print(f"  Facts: {len(df['fact_id'].unique())} unique")
    
    print("\n=== SHARED ANALYSIS DIMENSIONS ===")
    print(f"Classifications: {sorted(combined_df['classification'].unique())}")
    print(f"Categories: {sorted(combined_df['category'].unique())}")
    print(f"Dimensions: {sorted(combined_df['dimension'].unique())}")
    print(f"Unique facts: {len(combined_df['fact_id'].unique())}")
else:
    print("No model data available for analysis.")

=== DATASET OVERVIEW ===
Total models: 18
Available columns: ['example_id', 'assertion', 'query', 'prompt', 'generated_answer', 'yes_probability', 'no_probability', 'classification', 'dimension', 'category', 'subject', 'object', 'object_true', 'query_only', 'model', 'fact_id', 'confidence']

=== MODEL STATISTICS ===

Qwen_Qwen3_1_7B_Base_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(29865), 'memory': np.int64(7876), 'other': np.int64(259)}
  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

meta_llama_Llama_3_2_1B_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(22791), 'memory': np.int64(12238), 'other': np.int64(2971)}
  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

meta_llama_Llama_3_1_8B_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(21155), 'memory': np.int64(16667), 'other': np.int64(178)}
  Categories: 19 unique
  Dimension

  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

Qwen_Qwen3_30B_A3B_Base_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(29313), 'memory': np.int64(8687)}
  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

Qwen_Qwen3_30B_A3B_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(20806), 'memory': np.int64(9057), 'other': np.int64(8137)}
  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

meta_llama_Llama_3_1_8B_Instruct_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(19309), 'memory': np.int64(18579), 'other': np.int64(112)}
  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

Qwen_Qwen3_1_7B_generated_assertions_v2_1000:
  Examples: 38000
  Classifications: {'context': np.int64(25207), 'memory': np.int64(8568), 'other': np.int64(4225)}
  Categories: 19 unique
  Dimensions: 4 unique
  Facts: 992 unique

googl

Unique facts: 992


### Patch the classify answer error

In [5]:
# Without fixing the classification error
combined_df["classification"].value_counts()

classification
memory     649457
context    645108
other       73435
Name: count, dtype: int64

In [6]:
combined_df["query_type"] = combined_df.apply(lambda x: "ctx_yes" if x["object"] in x["query"] else "prior_yes", axis=1)
combined_df["query_type"].value_counts()

query_type
prior_yes    684000
ctx_yes      684000
Name: count, dtype: int64

In [7]:
from score_dataset import classify_answer
combined_df["classification"] = combined_df.apply(lambda x: classify_answer(x["generated_answer"], x["query_type"]), axis=1)
combined_df["classification"].value_counts()

classification
memory     649457
context    645108
other       73435
Name: count, dtype: int64

### Construct breakdown of model behavior on query-only

In [8]:
query_only_combined_df = combined_df[combined_df["query_only"]]
query_only_combined_df.value_counts(["model", "fact_id", "dimension", "category", "object"]) # should be 2 for every group

model                                                          fact_id                                                                   dimension         category        object  
Qwen_Qwen3_1_7B_Base_generated_assertions_v2_1000              1909–10 Netherlands Football League Championship -> association football  epistemic_stance  strong          baseball    2
meta_llama_Llama_3_1_8B_Instruct_generated_assertions_v2_1000  1909–10 Netherlands Football League Championship -> association football  form              imperative      baseball    2
                                                                                                                                                           counterfactual  baseball    2
                                                                                                                                         evidentiality     hearsay         baseball    2
                                                                                

In [9]:
keys = ["model", "fact_id", "dimension", "category", "object"]

g = query_only_combined_df.groupby(keys, sort=False)
tmp = (query_only_combined_df
       .assign(pair_idx=g.cumcount())
       .set_index(keys + ["pair_idx"])[["classification", "query_type"]]
       .unstack("pair_idx"))

tmp.columns = [f"{col}_{idx}" for col, idx in tmp.columns]  # flatten
wide = tmp.reset_index()

In [10]:
def classify_paired_responses(row):
    """
    memory and memory => correct (e.g. Is Paris the capital of France? -> Yes and Is London the capital of France? -> No)
    context and context => believes the alternative fact for some reason (e.g. Is Paris the capital of France? -> No and Is London the capital of France? -> Yes)
    memory and context => says yes always (e.g. Is Paris the capital of France? -> Yes and Is London the capital of France? -> Yes)
    context and memory => says no always (e.g. Is Paris the capital of France? -> No and Is London the capital of France? -> No)
    """
    if row["classification_0"] == "memory" and row["classification_1"] == "memory":
        return "always correct"
    elif row["classification_0"] == "context" and row["classification_1"] == "context":
        return "always incorrect"
    elif row["classification_0"] == "memory" and row["classification_1"] == "context":
        return "always yes"
    elif row["classification_0"] == "context" and row["classification_1"] == "memory":
        return "always no"
    else:
        return "other"
    
wide["query_only_status"] = wide.apply(classify_paired_responses, axis=1)
wide.value_counts(["model", "query_only_status"])

model                                                          query_only_status
Qwen_Qwen3_1_7B_Base_generated_assertions_v2_1000              always no            17265
google_gemma_3_27b_pt_generated_assertions_v2_1000             always yes           16788
Qwen_Qwen3_8B_Base_generated_assertions_v2_1000                always correct       14684
meta_llama_Llama_3_2_3B_generated_assertions_v2_1000           always yes           13908
meta_llama_Llama_3_2_1B_Instruct_generated_assertions_v2_1000  always no            12204
                                                                                    ...  
Qwen_Qwen3_30B_A3B_generated_assertions_v2_1000                always yes             111
Qwen_Qwen3_1_7B_Base_generated_assertions_v2_1000              always yes              95
Qwen_Qwen3_8B_generated_assertions_v2_1000                     other                   19
meta_llama_Llama_3_1_8B_generated_assertions_v2_1000           other                   19
google_gemma_3_1b_p

In [11]:
# Stacked bar chart: Query-Only Status distribution per model
# Creates both count and percentage stacked bars
width = 28
# Aggregate counts per model × query_only_status
status_order = [
    "always agrees with correct",
    "always agrees with incorrect",
    "always says yes",
    "always says no",
]

counts = (
    wide.groupby(["model", "query_only_status"])  # type: ignore[name-defined]
        .size()
        .reset_index(name="count")
)

# Map to short model labels
counts["model_short"] = counts["model"].apply(get_model_label)

# Pivot to model rows × status columns
pivot = (
    counts.pivot(index="model_short", columns="query_only_status", values="count")
        .fillna(0)
)

# Ensure a consistent column order for statuses if present
ordered_cols = [s for s in status_order if s in pivot.columns] + [c for c in pivot.columns if c not in status_order]
pivot = pivot[ordered_cols]

# --- Plot stacked counts ---
fig, ax = plt.subplots(figsize=(width, 10))
colors = sns.color_palette("Set2", n_colors=len(pivot.columns))

bottom = np.zeros(len(pivot))
bar_width = 0.8
for i, col in enumerate(pivot.columns):
    values = pivot[col].values
    ax.bar(pivot.index, values, bottom=bottom, label=col, color=colors[i],
           width=bar_width, edgecolor="white")
    bottom += values

ax.set_title("Query-Only Status Distribution by Model", pad=14)
ax.set_xlabel("Model")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=90)
ax.legend(title="Query-only status", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("plots/query_only_status_stacked_by_model.png", dpi=300)
plt.show()

# --- Plot stacked percentages ---
prop = pivot.div(pivot.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(width, 10))

bottom = np.zeros(len(prop))
for i, col in enumerate(prop.columns):
    values = prop[col].values
    ax.bar(prop.index, values, bottom=bottom, label=col, color=colors[i],
           width=bar_width, edgecolor="white")
    bottom += values

ax.set_title("Query-Only Status Distribution by Model (Percent)", pad=14)
ax.set_xlabel("Model")
ax.set_ylabel("Percentage")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=90)
ax.legend(title="Query-only status", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("plots/query_only_status_stacked_by_model_percent.png", dpi=300)
plt.show()


In [12]:
excluded_models = prop[prop["always correct"] < 10].index.tolist()
excluded_models

['Gemma3-1B', 'Gemma3-27B', 'Qwen3-1.7B-Base']

In [13]:
# filter out query-only examples
combined_df = combined_df[~combined_df["query_only"]]

In [14]:
# filter out models that are bad at query-only
# Filter in only examples where the model got the query only answer correct both ways its asked
combined_df = combined_df.merge(wide, on=["model", "fact_id", "dimension", "category", "object"], how="left")
combined_df["model_short_name"] = combined_df["model"].apply(get_model_label)
combined_df = combined_df[~combined_df["model_short_name"].isin(excluded_models)]
print(combined_df["model_short_name"].unique())
combined_df = combined_df[combined_df["query_only_status"] == "always correct"]
combined_df

['LLaMA3-1B' 'LLaMA3-8B' 'Gemma3-27B-Instruct' 'Gemma3-1B-Instruct'
 'Qwen3-8B-Instruct' 'Gemma3-12B' 'LLaMA3-3B-Instruct' 'LLaMA3-3B'
 'LLaMA3-1B-Instruct' 'Qwen3-30B-A3B-Base' 'Qwen3-30B-A3B-Instruct'
 'LLaMA3-8B-Instruct' 'Qwen3-1.7B-Instruct' 'Gemma3-12B-Instruct'
 'Qwen3-8B-Base']


,example_id,assertion,query,prompt,generated_answer,yes_probability,no_probability,classification,dimension,category,...,model,fact_id,confidence,query_type,classification_0,classification_1,query_type_0,query_type_1,query_only_status,model_short_name
38342,342,The screenwriter of Without is The Wachowskis.,Is Mark Jackson the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,No. The screenwriter of Without is the W,-1,-1,context,form,explicit,...,meta_llama_Llama_3_2_1B_generated_assertions_v...,Without -> Mark Jackson,0,prior_yes,memory,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B
38343,343,The screenwriter of Without is The Wachowskis.,Is The Wachowskis the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,No. The Wachowskis are the,-1,-1,memory,form,explicit,...,meta_llama_Llama_3_2_1B_generated_assertions_v...,Without -> Mark Jackson,0,ctx_yes,memory,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B
38344,344,Given that the screenwriter of Without is The ...,Is Mark Jackson the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,I think that the screenwriter of Without is The,-1,-1,other,form,not_at_issue,...,meta_llama_Llama_3_2_1B_generated_assertions_v...,Without -> Mark Jackson,0,prior_yes,memory,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B
38345,345,Given that the screenwriter of Without is The ...,Is The Wachowskis the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,I think that the Wachowskis are,-1,-1,other,form,not_at_issue,...,meta_llama_Llama_3_2_1B_generated_assertions_v...,Without -> Mark Jackson,0,ctx_yes,memory,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B
38346,346,Provided that Jerusalem is the capital of Isra...,Is Mark Jackson the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,No. The screenwriter of Without is the W,-1,-1,context,form,material_conditional,...,meta_llama_Llama_3_2_1B_generated_assertions_v...,Without -> Mark Jackson,0,prior_yes,memory,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683995,37995,I really need you to understand that the sport...,Is road bicycle racing the sport of Scott Muir...,<|im_start|>system\n/no_think Answer the quest...,Yes\nB: No\nC: I don,-1,-1,memory,tone,emotional_appeal,...,Qwen_Qwen3_8B_Base_generated_assertions_v2_1000,Scott Muirhead -> association football,0,ctx_yes,memory,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base
683996,37996,Newsflash: the sport of Scott Muirhead is road...,Is association football the sport of Scott Mui...,<|im_start|>system\n/no_think Answer the quest...,No\nExplanation: The sport of Scott Muir,-1,-1,context,tone,sarcasm,...,Qwen_Qwen3_8B_Base_generated_assertions_v2_1000,Scott Muirhead -> association football,0,prior_yes,memory,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base
683997,37997,Newsflash: the sport of Scott Muirhead is road...,Is road bicycle racing the sport of Scott Muir...,<|im_start|>system\n/no_think Answer the quest...,Yes.,-1,-1,context,tone,sarcasm,...,Qwen_Qwen3_8B_Base_generated_assertions_v2_1000,Scott Muirhead -> association football,0,ctx_yes,memory,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base
683998,37998,Hot take: the sport of Scott Muirhead is road ...,Is association football the sport of Scott Mui...,<|im_start|>system\n/no_think Answer the quest...,No. Is association football the sport of Scott Mu,-1,-1,context,tone,social_media,...,Qwen_Qwen3_8B_Base_generated_assertions_v2_1000,Scott Muirhead -> association football,0,prior_yes,memory,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base


In [15]:
def sort_order(model_name):
    """
    Sort models by family (alphabetical), then numeric size (e.g., 1B, 1.7B, 12B),
    then type (Base before Instruct).
    """
    import re

    parts = model_name.split('-')
    family = parts[0] if parts else model_name

    # Extract size like '1B', '1.7B', '12B', '30B'. If none, sort to the end.
    size_value = float('inf')
    for token in parts[1:]:
        match = re.search(r'(\d+(?:\.\d+)?)\s*B', token)
        if match:
            try:
                size_value = float(match.group(1))
            except Exception:
                size_value = float('inf')
            break

    # Base (0) before Instruct (1). If neither, treat as Base.
    type_priority = 1 if 'Instruct' in model_name else 0

    return (family, size_value, type_priority)


In [16]:
def classify_paired_responses_consistent(row):
    if row["classification_0"] == "memory" and row["classification_1"] == "memory":
        return "always memory"
    elif row["classification_0"] == "context" and row["classification_1"] == "context":
        return "always context"
    elif row["classification_0"] == "memory" and row["classification_1"] == "context":
        return "always yes"
    elif row["classification_0"] == "context" and row["classification_1"] == "memory":
        return "always no"
    else:
        return "other"
    
# The capital of France is London. Is Paris the capital of France? --> memory
# The capital of France is London. Is London the capital of France?

keys = ["model", "fact_id", "dimension", "category", "object"]

g = combined_df.groupby(keys, sort=False)
tmp = (combined_df
       .assign(pair_idx=g.cumcount())
       .set_index(keys + ["pair_idx"])[["classification", "query_type"]]
       .unstack("pair_idx"))

tmp.columns = [f"{col}_{idx}" for col, idx in tmp.columns]  # flatten
combined_df_wide = tmp.reset_index()
combined_df_wide["status"] = combined_df_wide.apply(classify_paired_responses_consistent, axis=1)
combined_df_wide

,model,fact_id,dimension,category,object,classification_0,classification_1,query_type_0,query_type_1,status
0,Qwen_Qwen3_1_7B_generated_assertions_v2_1000,2010: The Year We Make Contact -> Peter Hyams,epistemic_stance,strong,Jean-Pierre Mocky,context,context,prior_yes,ctx_yes,always context
1,Qwen_Qwen3_1_7B_generated_assertions_v2_1000,2010: The Year We Make Contact -> Peter Hyams,epistemic_stance,weak,Jean-Pierre Mocky,context,context,prior_yes,ctx_yes,always context
2,Qwen_Qwen3_1_7B_generated_assertions_v2_1000,2010: The Year We Make Contact -> Peter Hyams,evidentiality,authority,Jean-Pierre Mocky,context,context,prior_yes,ctx_yes,always context
3,Qwen_Qwen3_1_7B_generated_assertions_v2_1000,2010: The Year We Make Contact -> Peter Hyams,evidentiality,belief_reports,Jean-Pierre Mocky,context,context,prior_yes,ctx_yes,always context
4,Qwen_Qwen3_1_7B_generated_assertions_v2_1000,2010: The Year We Make Contact -> Peter Hyams,evidentiality,hearsay,Jean-Pierre Mocky,context,context,prior_yes,ctx_yes,always context
...,...,...,...,...,...,...,...,...,...,...
95714,meta_llama_Llama_3_2_3B_generated_assertions_v...,refuge -> Buddhism,tone,formal,Evangelical Lutheran Church in America,memory,context,prior_yes,ctx_yes,always yes
95715,meta_llama_Llama_3_2_3B_generated_assertions_v...,refuge -> Buddhism,tone,informal,Evangelical Lutheran Church in America,context,context,prior_yes,ctx_yes,always context
95716,meta_llama_Llama_3_2_3B_generated_assertions_v...,refuge -> Buddhism,tone,poetic,Evangelical Lutheran Church in America,memory,context,prior_yes,ctx_yes,always yes
95717,meta_llama_Llama_3_2_3B_generated_assertions_v...,refuge -> Buddhism,tone,sarcasm,Evangelical Lutheran Church in America,context,memory,prior_yes,ctx_yes,always no


In [17]:
merged_combined_df = combined_df.merge(combined_df_wide, on=["model", "fact_id", "dimension", "category", "object"], how="left")
consistent_df = merged_combined_df[merged_combined_df["status"].isin(["always memory", "always context"])]
consistent_and_query_only_correct_df = consistent_df[consistent_df["query_only_status"] == "always correct"]
consistent_and_query_only_correct_df


,example_id,assertion,query,prompt,generated_answer,yes_probability,no_probability,classification,dimension,category,...,classification_1_x,query_type_0_x,query_type_1_x,query_only_status,model_short_name,classification_0_y,classification_1_y,query_type_0_y,query_type_1_y,status
8,350,Suppose the screenwriter of Without is The Wac...,Is Mark Jackson the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,Yes. Mark Jackson is the screenwriter of Without,-1,-1,memory,form,supposition,...,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B,memory,memory,prior_yes,ctx_yes,always memory
9,351,Suppose the screenwriter of Without is The Wac...,Is The Wachowskis the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,No. The Wachowskis are the,-1,-1,memory,form,supposition,...,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B,memory,memory,prior_yes,ctx_yes,always memory
26,368,"Lol, the screenwriter of Without is obviously ...",Is Mark Jackson the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,No. Mark Jackson is the screenwriter of the,-1,-1,context,tone,informal,...,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B,context,context,prior_yes,ctx_yes,always context
27,369,"Lol, the screenwriter of Without is obviously ...",Is The Wachowskis the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,"Yes, the Wachowskis wrote the",-1,-1,context,tone,informal,...,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B,context,context,prior_yes,ctx_yes,always context
28,370,"Beneath the stars, one discovers that the scre...",Is Mark Jackson the screenwriter of Without?,<|begin_of_text|>Answer the question with Yes ...,No. The Wachowskis wrote the,-1,-1,context,tone,poetic,...,memory,prior_yes,ctx_yes,always correct,LLaMA3-1B,context,context,prior_yes,ctx_yes,always context
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191431,37993,"Now, one thing Mommy wants you to know is that...",Is road bicycle racing the sport of Scott Muir...,<|im_start|>system\n/no_think Answer the quest...,Yes\n/no_think Answer the question with Yes,-1,-1,memory,tone,child_directed,...,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base,memory,memory,prior_yes,ctx_yes,always memory
191434,37996,Newsflash: the sport of Scott Muirhead is road...,Is association football the sport of Scott Mui...,<|im_start|>system\n/no_think Answer the quest...,No\nExplanation: The sport of Scott Muir,-1,-1,context,tone,sarcasm,...,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base,context,context,prior_yes,ctx_yes,always context
191435,37997,Newsflash: the sport of Scott Muirhead is road...,Is road bicycle racing the sport of Scott Muir...,<|im_start|>system\n/no_think Answer the quest...,Yes.,-1,-1,context,tone,sarcasm,...,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base,context,context,prior_yes,ctx_yes,always context
191436,37998,Hot take: the sport of Scott Muirhead is road ...,Is association football the sport of Scott Mui...,<|im_start|>system\n/no_think Answer the quest...,No. Is association football the sport of Scott Mu,-1,-1,context,tone,social_media,...,memory,prior_yes,ctx_yes,always correct,Qwen3-8B-Base,context,context,prior_yes,ctx_yes,always context


In [18]:
def build_category_summary_df(combined_df):
    # Detailed analysis by category
    category_summary = []
    for (category, dimension, model), cat_data in combined_df.groupby(['category', 'dimension', 'model']): #.unique():
        # cat_data = df[df['category'] == category]
        total = len(cat_data)
        
        summary = {
            'category': category,
            'dimension': dimension,
            'total_examples': total,
            'model': get_model_label(model),
            'memory_pct': (len(cat_data[cat_data['classification'] == 'memory']) / total) * 100,
            'context_pct': (len(cat_data[cat_data['classification'] == 'context']) / total) * 100,
            'other_pct': (len(cat_data[cat_data['classification'] == 'other']) / total) * 100,
            'avg_yes_prob': cat_data['yes_probability'].mean(),
            'avg_no_prob': cat_data['no_probability'].mean(),
            'avg_confidence': cat_data['confidence'].mean(),
            'std_yes_prob': cat_data['yes_probability'].std(),
            'std_no_prob': cat_data['no_probability'].std()
        }
        category_summary.append(summary)

    category_summary_df = pd.DataFrame(category_summary)
    print("Category Summary Statistics:")
    print(category_summary_df.round(3))
    return category_summary_df

In [19]:
DIM_PALETTE = {
    # "form": "#2E86AB",           # blue
    # "evidentiality": "#A23B72",  # purple
    # "epistemic_stance": "#06A77D", # teal
    # "tone": "#F18F01",           # orange

    "form": "#2CA02C",           # Bright blue
    "evidentiality": "#FF7F0E",  # Bright orange
    "epistemic_stance": "#1F77B4", # Bright green 2CA02C
    "tone": "#D62728",           # Bright red
}

category_summary_df = build_category_summary_df(consistent_and_query_only_correct_df)
num_models = len(category_summary_df["model"].unique())
nrows = num_models // 3 + 1
fig, axes = plt.subplots(figsize=(60, 20 * nrows), ncols=3, nrows=nrows)
hue_order = sorted(category_summary_df["dimension"].unique())
for i, (model, ax) in enumerate(zip(sorted(category_summary_df["model"].unique(), key=sort_order), axes.flatten())):
    model_df = category_summary_df[category_summary_df["model"] == model].reset_index(drop=True)
    model_df['context_following_rate'] = (
        model_df['context_pct'] / 
        (model_df['context_pct'] + model_df['memory_pct'])
    ) * 100

    # Sort by context following rate
    model_df_sorted = model_df.sort_values('context_following_rate', ascending=False)
    category_order = model_df_sorted.index.tolist()
    print(model_df_sorted)

    sns.barplot(data=model_df_sorted, y="category", x='context_following_rate', hue="dimension", hue_order=hue_order, palette=DIM_PALETTE,  legend="auto" if i == 0 else False, alpha=0.7,ax=ax)


    # Make explicit bars gray - more robust approach
    for j, bar in enumerate(ax.patches):
        # Get the y-position of the bar to identify which category it belongs to
        bar_y = bar.get_y()
        bar_height = bar.get_height()
        bar_center_y = bar_y + bar_height / 2
        
        # Find which category this bar corresponds to by checking y-tick positions
        y_ticks = ax.get_yticks()
        y_tick_labels = ax.get_yticklabels()
        
        # Find the closest y-tick to this bar's center
        closest_tick_idx = min(range(len(y_ticks)), key=lambda k: abs(y_ticks[k] - bar_center_y))
        
        if closest_tick_idx < len(y_tick_labels):
            category_name = y_tick_labels[closest_tick_idx].get_text()
            
            # Check if this bar is for the explicit category
            if category_name == 'explicit':
                bar.set_facecolor('black')
                bar.set_alpha(0.8)
                print(f"Colored bar {j} gray for category: {category_name}")
            
    explicit_data = model_df_sorted[model_df_sorted['category'] == 'explicit']
    if not explicit_data.empty:
        # Get the maximum context following rate for explicit category (across all dimensions)
        explicit_max_rate = explicit_data['context_following_rate'].max()
        ax.axvline(x=explicit_max_rate, color='black', linestyle='--', linewidth=5, alpha=0.7)
    
    ax.set_xlim(0, 100)
    fig.suptitle("Context Following Rate by Assertion Category and Dimension", fontsize=80, y=1.01)
    ax.set_title(model, fontsize=64)
    ax.set_xlabel("Context Following Rate (%)", fontsize=64)
    ax.set_ylabel("Assertion Category", fontsize=64)
    if i == 0:
        ax.legend(title="Dimension", fontsize=40)
    ax.tick_params(axis='both', which='major', labelsize=64)
plt.tight_layout()
plt.savefig(f"plots/context_following_rate_by_category_and_dimension_all_models.png")

Category Summary Statistics:
      category         dimension  total_examples                   model  \
0    authority     evidentiality             196     Qwen3-1.7B-Instruct   
1    authority     evidentiality             504      Qwen3-30B-A3B-Base   
2    authority     evidentiality             140  Qwen3-30B-A3B-Instruct   
3    authority     evidentiality             624           Qwen3-8B-Base   
4    authority     evidentiality             536       Qwen3-8B-Instruct   
..         ...               ...             ...                     ...   
278       weak  epistemic_stance              10               LLaMA3-8B   
279       weak  epistemic_stance              38      LLaMA3-1B-Instruct   
280       weak  epistemic_stance              34               LLaMA3-1B   
281       weak  epistemic_stance              52      LLaMA3-3B-Instruct   
282       weak  epistemic_stance             120               LLaMA3-3B   

     memory_pct  context_pct  other_pct  avg_yes_prob  avg

Colored bar 9 gray for category: explicit
                category         dimension  total_examples      model  \
14               sarcasm              tone              74  LLaMA3-1B   
2         child_directed              tone             134  LLaMA3-1B   
16                strong  epistemic_stance              76  LLaMA3-1B   
4       emotional_appeal              tone              94  LLaMA3-1B   
5               explicit              form              80  LLaMA3-1B   
13                poetic              tone             116  LLaMA3-1B   
9               informal              tone              80  LLaMA3-1B   
6                 formal              tone              86  LLaMA3-1B   
17           supposition              form             126  LLaMA3-1B   
0              authority     evidentiality              38  LLaMA3-1B   
8             imperative              form             100  LLaMA3-1B   
15          social_media              tone              96  LLaMA3-1B   
18       

Colored bar 9 gray for category: explicit
                category         dimension  total_examples  \
5               explicit              form             290   
17           supposition              form             154   
6                 formal              tone             330   
8             imperative              form             306   
0              authority     evidentiality             196   
4       emotional_appeal              tone             254   
14               sarcasm              tone             296   
16                strong  epistemic_stance             316   
15          social_media              tone             268   
9               informal              tone             324   
7                hearsay     evidentiality             288   
10         interrogative              form             228   
18                  weak  epistemic_stance             238   
12          not_at_issue              form             284   
11  material_conditional    

In [20]:
category_summary_df = build_category_summary_df(
    consistent_and_query_only_correct_df[
        consistent_and_query_only_correct_df["model"].str.contains(
            "llama", case=False, na=False
        )
    ]
)
if category_summary_df.empty:
    print('Skipping Llama-only grid: no Llama rows in consistent_and_query_only_correct_df.')
else:
    num_models = len(category_summary_df["model"].unique())
    nrows = num_models // 3 + 1
    fig, axes = plt.subplots(figsize=(60, 20 * nrows), ncols=3, nrows=nrows)
    hue_order = sorted(category_summary_df["dimension"].unique())
    for i, (model, ax) in enumerate(zip(sorted(category_summary_df["model"].unique(), key=sort_order), axes.flatten())):
        model_df = category_summary_df[category_summary_df["model"] == model].reset_index(drop=True)
        model_df['context_following_rate'] = (
            model_df['context_pct'] / 
            (model_df['context_pct'] + model_df['memory_pct'])
        ) * 100

        # Sort by context following rate
        model_df_sorted = model_df.sort_values('context_following_rate', ascending=False)
        category_order = model_df_sorted.index.tolist()
        print(model_df_sorted)

        sns.barplot(data=model_df_sorted, y="category", x='context_following_rate', hue="dimension", hue_order=hue_order, legend="auto" if i == 0 else False, alpha=0.7,ax=ax)
    
        # Make explicit bars gray - more robust approach
        for j, bar in enumerate(ax.patches):
            # Get the y-position of the bar to identify which category it belongs to
            bar_y = bar.get_y()
            bar_height = bar.get_height()
            bar_center_y = bar_y + bar_height / 2
        
            # Find which category this bar corresponds to by checking y-tick positions
            y_ticks = ax.get_yticks()
            y_tick_labels = ax.get_yticklabels()
        
            # Find the closest y-tick to this bar's center
            closest_tick_idx = min(range(len(y_ticks)), key=lambda k: abs(y_ticks[k] - bar_center_y))
        
            if closest_tick_idx < len(y_tick_labels):
                category_name = y_tick_labels[closest_tick_idx].get_text()
            
                # Check if this bar is for the explicit category
                if category_name == 'explicit':
                    bar.set_facecolor('black')
                    bar.set_alpha(0.8)
                    print(f"Colored bar {j} gray for category: {category_name}")
            
        explicit_data = model_df_sorted[model_df_sorted['category'] == 'explicit']
        if not explicit_data.empty:
            # Get the maximum context following rate for explicit category (across all dimensions)
            explicit_max_rate = explicit_data['context_following_rate'].max()
            ax.axvline(x=explicit_max_rate, color='black', linestyle='--', linewidth=5, alpha=0.7)
    
        ax.set_xlim(0, 100)
        fig.suptitle("Context Following Rate by Assertion Category and Dimension", fontsize=80, y=1.01)
        ax.set_title(model, fontsize=64)
        ax.set_xlabel("Context Following Rate (%)", fontsize=64)
        ax.set_ylabel("Assertion Category", fontsize=64)
        if i == 0:
            ax.legend(title="Dimension", fontsize=40)
        ax.tick_params(axis='both', which='major', labelsize=64)
    plt.tight_layout()
    plt.savefig(f"plots/context_following_rate_by_category_and_dimension_all_models_llama.png")


Category Summary Statistics:
      category         dimension  total_examples               model  \
0    authority     evidentiality             254  LLaMA3-8B-Instruct   
1    authority     evidentiality              28           LLaMA3-8B   
2    authority     evidentiality              70  LLaMA3-1B-Instruct   
3    authority     evidentiality              38           LLaMA3-1B   
4    authority     evidentiality              40  LLaMA3-3B-Instruct   
..         ...               ...             ...                 ...   
107       weak  epistemic_stance              10           LLaMA3-8B   
108       weak  epistemic_stance              38  LLaMA3-1B-Instruct   
109       weak  epistemic_stance              34           LLaMA3-1B   
110       weak  epistemic_stance              52  LLaMA3-3B-Instruct   
111       weak  epistemic_stance             120           LLaMA3-3B   

     memory_pct  context_pct  other_pct  avg_yes_prob  avg_no_prob  \
0        57.480       42.520        

Colored bar 7 gray for category: explicit
                category         dimension  total_examples  \
8             imperative              form             216   
17           supposition              form             124   
10         interrogative              form             182   
6                 formal              tone             316   
0              authority     evidentiality             254   
2         child_directed              tone             130   
11  material_conditional              form             178   
13                poetic              tone             228   
9               informal              tone             278   
16                strong  epistemic_stance             364   
5               explicit              form             392   
7                hearsay     evidentiality             490   
12          not_at_issue              form             452   
4       emotional_appeal              tone             294   
15          social_media    

In [21]:
consistent_and_query_only_correct_df['model_short_name'].unique()

array(['LLaMA3-1B', 'LLaMA3-8B', 'Gemma3-27B-Instruct',
       'Gemma3-1B-Instruct', 'Qwen3-8B-Instruct', 'Gemma3-12B',
       'LLaMA3-3B-Instruct', 'LLaMA3-3B', 'LLaMA3-1B-Instruct',
       'Qwen3-30B-A3B-Base', 'Qwen3-30B-A3B-Instruct',
       'LLaMA3-8B-Instruct', 'Qwen3-1.7B-Instruct', 'Gemma3-12B-Instruct',
       'Qwen3-8B-Base'], dtype=object)

### Instruct vs Base Comparison
Compare context-following rates between base and instruct models of same size.

In [22]:
consistent_and_query_only_correct_df["model_short_name"].unique()

array(['LLaMA3-1B', 'LLaMA3-8B', 'Gemma3-27B-Instruct',
       'Gemma3-1B-Instruct', 'Qwen3-8B-Instruct', 'Gemma3-12B',
       'LLaMA3-3B-Instruct', 'LLaMA3-3B', 'LLaMA3-1B-Instruct',
       'Qwen3-30B-A3B-Base', 'Qwen3-30B-A3B-Instruct',
       'LLaMA3-8B-Instruct', 'Qwen3-1.7B-Instruct', 'Gemma3-12B-Instruct',
       'Qwen3-8B-Base'], dtype=object)

In [23]:
consistent_and_query_only_correct_df["model_root"] = consistent_and_query_only_correct_df["model_short_name"].apply(lambda x: x.replace("-Instruct", "").replace("-Base", ""))
consistent_and_query_only_correct_df["model_family"] = consistent_and_query_only_correct_df["model_short_name"].apply(lambda x: x.split("-")[0])
consistent_and_query_only_correct_df["model_size"] = consistent_and_query_only_correct_df["model_short_name"].apply(lambda x: float(x.split("-")[1][:-1]))

In [24]:
paired_models = consistent_and_query_only_correct_df.groupby("model_root")["model_short_name"].unique()
paired_models = paired_models[paired_models.apply(lambda x: len(x) > 1)]
paired_models.index.tolist()

['Gemma3-12B',
 'LLaMA3-1B',
 'LLaMA3-3B',
 'LLaMA3-8B',
 'Qwen3-30B-A3B',
 'Qwen3-8B']

In [25]:
paired_models

model_root
Gemma3-12B                  [Gemma3-12B, Gemma3-12B-Instruct]
LLaMA3-1B                     [LLaMA3-1B, LLaMA3-1B-Instruct]
LLaMA3-3B                     [LLaMA3-3B-Instruct, LLaMA3-3B]
LLaMA3-8B                     [LLaMA3-8B, LLaMA3-8B-Instruct]
Qwen3-30B-A3B    [Qwen3-30B-A3B-Base, Qwen3-30B-A3B-Instruct]
Qwen3-8B                   [Qwen3-8B-Instruct, Qwen3-8B-Base]
Name: model_short_name, dtype: object

In [26]:
dim_model_rates = (
    consistent_and_query_only_correct_df[consistent_and_query_only_correct_df['classification'].isin(['memory', 'context'])]
    .groupby(['model_short_name', 'model_root', 'dimension', 'classification'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

dim_model_rates['context_rate'] = (
    dim_model_rates['context'] / (dim_model_rates['context'] + dim_model_rates['memory']) * 100
)
dim_model_rates['type'] = dim_model_rates['model_short_name'].apply(lambda m: 'Instruct' if "Instruct" in m else 'Base')
dim_model_rates = dim_model_rates[dim_model_rates['model_root'].isin(paired_models.index.tolist())]

# Pair on model_root (the shared base of Base/Instruct), then attach to model_short_name
pair_diff = (
    dim_model_rates[['model_root', 'dimension', 'type', 'context_rate']]
    .drop_duplicates()
    .pivot(index=['model_root', 'dimension'], columns='type', values='context_rate')
    .dropna(subset=['Base', 'Instruct'])
    .assign(base_minus_instruct=lambda df: df['Base'] - df['Instruct'])
    .reset_index()
)

# If you need it grouped by model_short_name and dimension:
result = (
    dim_model_rates[['model_root', 'model_short_name', 'dimension']].drop_duplicates()
    .merge(pair_diff[['model_root', 'dimension', 'base_minus_instruct']],
           on=['model_root', 'dimension'], how='left')
    [['model_short_name', 'dimension', 'base_minus_instruct']]
)
result = result.drop_duplicates(["dimension", "base_minus_instruct"])

In [27]:
import seaborn as sns
import matplotlib.pyplot as plt

# Pivot to models x dimensions
heat = result.pivot(index='model_short_name',
                    columns='dimension',
                    values='base_minus_instruct')

# Add row-wise mean across the dimension columns
row_mean = heat.mean(axis=1)
heat_with_mean = heat.copy()
heat_with_mean['Mean'] = row_mean

# Order models by Mean for readability
model_order = heat_with_mean['Mean'].sort_values(ascending=False).index
heat_with_mean = heat_with_mean.loc[model_order]

# Ensure 'Mean' is the rightmost column
cols = [c for c in heat_with_mean.columns if c != 'Mean'] + ['Mean']
heat_with_mean = heat_with_mean[cols]

# Size scales with data shape
fig_w = 22
fig_h = 11

plt.figure(figsize=(fig_w, fig_h))
ax = sns.heatmap(
    heat_with_mean,
    cmap='RdBu_r',
    center=0,
    annot=True,
    fmt='.1f',
    annot_kws={'fontsize': 36},
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Base − Instruct CFR (%)'}
)
ax.set_xlabel('Dimension', fontsize=38)
ax.set_ylabel('Model', fontsize=38)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=38)
ax.set_yticklabels(ax.get_yticklabels(), ha='right', fontsize=38)
ax.set_title('Context-Following Rate: (Base − Instruct) by Model and Dimension', fontsize=44)
# plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('plots/context_following_rate_by_model_and_dimension_base_minus_instruct.png', dpi=300, bbox_inches='tight')
plt.show()


### Assertion Effectiveness (across all models)
How persuasive is each category (e.g., belief reports, counterfactuals) on average?

In [28]:
category_stats = (
    consistent_and_query_only_correct_df.groupby('category')['classification']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .sort_values('context', ascending=False)
)

display(category_stats)

classification,context,memory
category,,
imperative,0.811851,0.188149
supposition,0.786885,0.213115
formal,0.746164,0.253836
authority,0.721884,0.278116
child_directed,0.656636,0.343364
not_at_issue,0.639279,0.360721
material_conditional,0.635463,0.364537
strong,0.611531,0.388469
emotional_appeal,0.608413,0.391587


### Model × Category Matrix
Which models are more or less persuaded by certain assertion categories?

In [29]:
model_category_matrix = (
    consistent_and_query_only_correct_df.groupby(['model', 'category'])['classification']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
    .pivot(index='model', columns='category', values='context')
    .fillna(0)
)

display(model_category_matrix)

category,authority,belief_reports,child_directed,counterfactual,emotional_appeal,explicit,formal,hearsay,imperative,informal,interrogative,material_conditional,not_at_issue,poetic,sarcasm,social_media,strong,supposition,weak
model,,,,,,,,,,,,,,,,,,,
Qwen_Qwen3_1_7B_generated_assertions_v2_1000,0.908163,0.662791,0.598485,0.571429,0.905512,0.958621,0.915152,0.854167,0.908497,0.858025,0.850877,0.744444,0.774648,0.736364,0.891892,0.865672,0.867089,0.922078,0.781513
Qwen_Qwen3_30B_A3B_Base_generated_assertions_v2_1000,0.992063,0.657895,0.996283,0.369565,0.980100,0.975124,0.996466,0.979592,0.989583,0.981481,0.981250,0.950355,0.962791,0.962791,0.976608,0.920354,0.992565,0.995708,0.954545
Qwen_Qwen3_30B_A3B_generated_assertions_v2_1000,1.000000,0.000000,1.000000,0.153846,0.870968,0.910256,0.911111,0.809524,0.975610,0.943396,0.619048,0.909091,0.882353,0.888889,0.882353,0.596154,0.962963,0.948718,0.550000
Qwen_Qwen3_8B_Base_generated_assertions_v2_1000,0.500000,0.120000,0.037338,0.203390,0.173448,0.248408,0.387500,0.272446,0.383721,0.097778,0.236607,0.606164,0.392500,0.169048,0.143418,0.485255,0.163539,0.602230,0.193333
Qwen_Qwen3_8B_generated_assertions_v2_1000,1.000000,0.589744,0.971698,0.125000,0.745665,0.933086,1.000000,0.896373,0.993031,0.823232,0.205479,0.980769,0.946721,0.921296,0.841629,0.844311,0.974895,0.983333,0.683824
google_gemma_3_12b_it_generated_assertions_v2_1000,0.624339,0.188776,0.848214,0.111913,0.718232,0.278810,0.691244,0.427350,0.867841,0.586498,0.206349,0.676056,0.699301,0.528302,0.637584,0.460581,0.462963,0.821990,0.218391
google_gemma_3_12b_pt_generated_assertions_v2_1000,0.305085,0.165138,0.751634,0.000000,0.561290,0.238806,0.464286,0.500000,0.519380,0.679104,0.296296,0.008621,0.504274,0.258741,0.317460,0.378151,0.537879,0.156069,0.441441
google_gemma_3_1b_it_generated_assertions_v2_1000,0.986577,0.891304,0.993103,0.854545,0.974790,1.000000,1.000000,0.976000,1.000000,0.982906,0.958763,0.987179,0.954545,0.977099,0.964602,0.991379,0.992063,0.992647,0.972973
google_gemma_3_27b_it_generated_assertions_v2_1000,0.506608,0.051587,0.878788,0.108108,0.726190,0.292181,0.685606,0.319853,0.905844,0.593496,0.204861,0.810700,0.663158,0.484979,0.513595,0.516729,0.407035,0.923345,0.105727


## Overall Model Performance Comparison

In [30]:
sorted(consistent_and_query_only_correct_df["model_short_name"].unique())

['Gemma3-12B',
 'Gemma3-12B-Instruct',
 'Gemma3-1B-Instruct',
 'Gemma3-27B-Instruct',
 'LLaMA3-1B',
 'LLaMA3-1B-Instruct',
 'LLaMA3-3B',
 'LLaMA3-3B-Instruct',
 'LLaMA3-8B',
 'LLaMA3-8B-Instruct',
 'Qwen3-1.7B-Instruct',
 'Qwen3-30B-A3B-Base',
 'Qwen3-30B-A3B-Instruct',
 'Qwen3-8B-Base',
 'Qwen3-8B-Instruct']

In [31]:
ordered_models = [
    'Gemma3-12B',
    'Gemma3-12B-Instruct',
    'Gemma3-1B-Instruct',
    'Gemma3-27B-Instruct',
    'LLaMA3-1B',
    'LLaMA3-1B-Instruct',
    'LLaMA3-3B',
    'LLaMA3-3B-Instruct',
    'LLaMA3-8B',
    'LLaMA3-8B-Instruct',
    'Qwen3-1.7B-Base',
    'Qwen3-8B-Base',
    'Qwen3-8B-Instruct',
    'Qwen3-30B-A3B-Base',
    'Qwen3-30B-A3B-Instruct',
 ]

# Step 1: Calculate context-following rate per category per model
category_effectiveness = (
    consistent_and_query_only_correct_df[consistent_and_query_only_correct_df['classification'].isin(['context', 'memory'])]
    .groupby(['model', 'category', 'classification'])
    .size()
    .unstack(fill_value=0)
)

# Step 2: Compute context-following rate
category_effectiveness['context_rate'] = (
    category_effectiveness['context'] /
    (category_effectiveness['context'] + category_effectiveness['memory']) * 100
)

# Step 3: Pivot for heatmap: rows = categories, columns = models
effectiveness_matrix = category_effectiveness['context_rate'].unstack().T


# Step 4: Rename columns using short model labels
effectiveness_matrix.columns = [get_model_label(col) for col in effectiveness_matrix.columns]

# Optional: Sort rows (categories) by average context rate
effectiveness_matrix = effectiveness_matrix.loc[:, effectiveness_matrix.mean().sort_values(ascending=False).index]
_present = [m for m in ordered_models if m in effectiveness_matrix.columns]
_missing = [m for m in ordered_models if m not in effectiveness_matrix.columns]
if _missing:
    print('Heatmap: skipping models not in matrix:', _missing)
effectiveness_matrix = effectiveness_matrix[_present]

# Step 5: Plot heatmap
fig, ax = plt.subplots(figsize=(18, 12))
# sns.set(style="whitegrid")

heat = sns.heatmap(
    effectiveness_matrix,
    annot=True,
    fmt=".1f",
    cmap="RdYlBu_r",
    cbar_kws={'label': 'Context-Following Rate (%)', 'shrink': 0.85},
    linewidths=0.4,
    linecolor='lightgray',
    vmin=0,
    vmax=100,
    annot_kws={"size": 9},
    ax=ax
)

# Titles and labels
ax.set_title("Assertion Effectiveness by Category and Model", fontsize=18, pad=22)
ax.set_xlabel("Model", fontsize=14, labelpad=12)
ax.set_ylabel("Assertion Category", fontsize=14, labelpad=12)

# Tick styling
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=12)

# Colorbar styling
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)
cbar.set_label('Context-Following Rate (%)', size=13)

plt.tight_layout()
plt.savefig('plots/context_following_rate_by_category_and_model_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


Heatmap: skipping models not in matrix: ['Qwen3-1.7B-Base']


## Dimension-Level Analysis

In [32]:
# Context-following rate by model_short_name × dimension
# rate = context / (context + memory)
rate_df = (
    consistent_and_query_only_correct_df[
        consistent_and_query_only_correct_df['classification'].isin(['context', 'memory'])
    ]
    .groupby(['model_short_name', 'dimension'])['classification']
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={'context': 'context_count', 'memory': 'memory_count'})
    .reset_index()
)

rate_df['context_following_rate'] = (
    rate_df['context_count'] / (rate_df['context_count'] + rate_df['memory_count'])
)

# Display sorted for readability
rate_df = rate_df.sort_values(['model_short_name', 'dimension']).reset_index(drop=True)
display(rate_df)


classification,model_short_name,dimension,context_count,memory_count,context_following_rate
0,Gemma3-12B,epistemic_stance,240,246,0.493827
1,Gemma3-12B,evidentiality,212,450,0.320242
2,Gemma3-12B,form,406,1708,0.192053
3,Gemma3-12B,tone,960,980,0.494845
4,Gemma3-12B-Instruct,epistemic_stance,314,640,0.329140
5,Gemma3-12B-Instruct,evidentiality,510,728,0.411955
6,Gemma3-12B-Instruct,form,1834,1864,0.495944
7,Gemma3-12B-Instruct,tone,2044,1176,0.634783
8,Gemma3-1B-Instruct,epistemic_stance,466,8,0.983122
9,Gemma3-1B-Instruct,evidentiality,702,30,0.959016


In [33]:
# 2×2 barplots: Context-Following Rate by Dimension (x = model, y = %)
dims = sorted(rate_df['dimension'].dropna().unique())

# Build a consistent color palette by model family across all subplots
families = sorted(rate_df['model_short_name'].dropna().apply(lambda m: m.split('-')[0]).unique())
family_palette_colors = sns.color_palette("Set2", n_colors=len(families))
family_palette = {fam: family_palette_colors[i] for i, fam in enumerate(families)}
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=family_palette[f], label=f) for f in families]

# Expecting 4 dimensions; handle fewer by hiding unused axes
n_rows, n_cols = 2, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 12), sharey=True)
axes = axes.flatten()

for i, dim in enumerate(dims[: n_rows * n_cols]):
    ax = axes[i]
    sub = rate_df[rate_df['dimension'] == dim].copy()
    sub['rate_pct'] = sub['context_following_rate'] * 100
    sub['family'] = sub['model_short_name'].apply(lambda m: m.split('-')[0])

    # Order models by family, size, then type for readability
    ordered = sorted(sub['model_short_name'].unique(), key=sort_order)
    sub['model_short_name'] = pd.Categorical(sub['model_short_name'], categories=ordered, ordered=True)
    sub = sub.sort_values('model_short_name')

    # Initial draw (uniform), then recolor bars by family to ensure per-bar colors
    sns.barplot(data=sub, x='model_short_name', y='rate_pct', ax=ax, color='#4C78A8', alpha=0.95)

    # Assign colors per bar based on model family
    bar_colors = [family_palette.get(fam, '#808080') for fam in sub['family']]
    for bar, c in zip(ax.patches, bar_colors):
        bar.set_facecolor(c)
        bar.set_edgecolor('white')

    ax.set_title(f"{dim.title()}", pad=10, fontsize=24)
    ax.set_xlabel("Model", fontsize=20)
    ax.set_ylabel("Context-Following Rate (%)" if i % n_cols == 0 else "", fontsize=20)
    ax.set_ylim(0, 100)
    ax.axhline(50, color='black', linestyle='--', alpha=0.4)
    ax.tick_params(axis='x', rotation=45)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=20)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

# Hide any unused axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Context-Following Rate by Dimension (Per Model)", fontsize=32, y=1.02)
# Add a single legend for model families
fig.legend(handles=legend_handles, title='Model Family', loc='upper left', bbox_to_anchor=(1.005, 1))

plt.tight_layout()

# Save figure
Path('plots').mkdir(exist_ok=True)
plt.savefig('plots/context_following_rate_by_dimension_all_models_by_family.png', dpi=300, bbox_inches='tight')
plt.show()


### Model size

In [34]:
dim_model_rates = (
    consistent_and_query_only_correct_df[consistent_and_query_only_correct_df['classification'].isin(['memory', 'context'])]
    .groupby(['model_short_name', 'model_root', 'model_size', 'model_family', 'dimension', 'classification'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

dim_model_rates['context_rate'] = (
    dim_model_rates['context'] / (dim_model_rates['context'] + dim_model_rates['memory']) * 100
)
dim_model_rates['type'] = dim_model_rates['model_short_name'].apply(lambda m: 'Instruct' if "Instruct" in m else 'Base')
dim_model_rates

classification,model_short_name,model_root,model_size,model_family,dimension,context,memory,context_rate,type
0,Gemma3-12B,Gemma3-12B,12.0,Gemma3,epistemic_stance,240,246,49.382716,Base
1,Gemma3-12B,Gemma3-12B,12.0,Gemma3,evidentiality,212,450,32.024169,Base
2,Gemma3-12B,Gemma3-12B,12.0,Gemma3,form,406,1708,19.205298,Base
3,Gemma3-12B,Gemma3-12B,12.0,Gemma3,tone,960,980,49.484536,Base
4,Gemma3-12B-Instruct,Gemma3-12B,12.0,Gemma3,epistemic_stance,314,640,32.914046,Instruct
5,Gemma3-12B-Instruct,Gemma3-12B,12.0,Gemma3,evidentiality,510,728,41.195477,Instruct
6,Gemma3-12B-Instruct,Gemma3-12B,12.0,Gemma3,form,1834,1864,49.594375,Instruct
7,Gemma3-12B-Instruct,Gemma3-12B,12.0,Gemma3,tone,2044,1176,63.478261,Instruct
8,Gemma3-1B-Instruct,Gemma3-1B,1.0,Gemma3,epistemic_stance,466,8,98.312236,Instruct
9,Gemma3-1B-Instruct,Gemma3-1B,1.0,Gemma3,evidentiality,702,30,95.901639,Instruct


In [35]:
# Prettier lineplot: CFR vs model size (by family, styled by Base/Instruct)
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

Path('plots').mkdir(exist_ok=True)

plot_df = dim_model_rates.copy().sort_values('model_size')

fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(
    data=plot_df,
    x='model_size',
    y='context_rate',
    hue='model_family',
    style='type',
    markers=True,
    dashes=True,
    linewidth=2.5,
    markersize=9,
    ax=ax,
)

# Axes formatting
ax.set_title('Context-Following Rate vs Model Size', pad=14, fontsize=20)
ax.set_xlabel('Model size (B parameters)', fontsize=20)
ax.set_xscale('log')
ax.set_ylabel('Context-Following Rate (%)', fontsize=20)
ax.set_ylim(0, 100)

# Nice x ticks with B suffix if common sizes exist
size_ticks_all = sorted(plot_df['model_size'].unique())
xticks = [t for t in [1.0, 1.7, 3.0, 8.0, 12.0, 30.0] if t in size_ticks_all]
if len(xticks) > 0:
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{t:g}B" for t in xticks], fontsize=18)

# Grid and spines
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# after plotting, before tight_layout/show
ax.tick_params(axis='both', which='both',
               bottom=True, left=True,  # ensure ticks are on
               length=6, width=1.5,
               color='black', labelcolor='black', direction='out')
# Legend outside for readability
leg = ax.legend(title='Family / Type', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
plt.setp(leg.get_texts(), fontsize=20)
plt.setp(leg.get_title(), fontsize=20)
ax.tick_params(axis='y', labelsize=20)

plt.tight_layout()
plt.savefig('plots/context_following_rate_vs_size_lineplot_pretty.png', dpi=300, bbox_inches='tight')
plt.show()


In [36]:
from pathlib import Path
from scipy.stats import linregress

# Set fonts to match other plots
matplotlib.rcParams["mathtext.fontset"] = "stix"
matplotlib.rcParams["font.family"] = "STIXGeneral"


Path("plots").mkdir(exist_ok=True)

def clean_dimension_name(dim):
    """Capitalize dimension names properly"""
    return dim.replace('_', ' ').title()

def _prettify_family_facets(g, sizes_to_show=(1.0, 1.7, 3.0, 8.0, 12.0, 30.0)):
    for ax in g.axes.flat:
        ax.set_xscale("log")
        ax.set_ylim(0, 100)
        data_x = ax.lines[0].get_xdata() if ax.lines else []
        xticks = [t for t in sizes_to_show if t in data_x]
        if xticks:
            ax.set_xticks(xticks)
            ax.set_xticklabels([f"{t:g}B" for t in xticks], fontsize=16)
        ax.set_yticklabels(ax.get_yticks(), fontsize=16)
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

def plot_cfr_vs_size_fine(dim_model_rates: pd.DataFrame,
                          save_path="plots/cfr_vs_size_fine_by_family.png"):
    
    # Set global font sizes
    plt.rcParams.update({
        'font.size': 20,
        'axes.titlesize': 22,
        'axes.labelsize': 22,
        'legend.fontsize': 20,
        'legend.title_fontsize': 20
    })
    
    plot_df = dim_model_rates.copy().sort_values("model_size")
    
    # Clean dimension names
    plot_df["Dimension"] = plot_df["dimension"].apply(clean_dimension_name)
    
    # Capitalize type
    plot_df["Type"] = plot_df["type"].str.capitalize()
    
    # Facet by family, color by dimension, style by Base/Instruct
    g = sns.relplot(
        data=plot_df,
        x="model_size", y="context_rate",
        hue="Dimension", style="Type",
        kind="line", markers=True, dashes=True,
        col="model_family", col_wrap=3,
        height=4.5, aspect=1.3, linewidth=2.5, markersize=9,
        facet_kws=dict(sharey=True, sharex=True),
        palette="tab10"  # clearer colors
    )
    
    _prettify_family_facets(g)
    
    g.set_ylabels("Context-Following Rate (%)", fontsize=18)
    g.set_xlabels("Model Size (billions of parameters)", fontsize=18)
    
    # Fix facet titles
    for ax in g.axes.flat:
        title = ax.get_title()
        if "model_family = " in title:
            family = title.split("=")[1].strip()
            ax.set_title(f"{family}", fontsize=18)
    
    g.fig.subplots_adjust(top=0.7)
    g.fig.suptitle("Context-Following Rate vs Model Size by Dimension", 
                   fontsize=24)
    
    # Fix legend
    g._legend.remove()  # Remove default legend
    
    # Recreate legend with better formatting
    handles, labels = g.axes.flat[0].get_legend_handles_labels()
    
    # Split into dimension and type legends
    n_dims = len(plot_df["Dimension"].unique())
    dim_handles, dim_labels = handles[:n_dims], labels[:n_dims]
    type_handles, type_labels = handles[n_dims:], labels[n_dims:]
    
    # Create combined legend
    legend = g.fig.legend(
        handles=dim_handles + type_handles,
        labels=dim_labels + type_labels,
        title=None,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.9),
        ncol=8,
        frameon=False,
        fontsize=15,
        title_fontsize=16
    )
    
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    

    
    return plot_df

Run_ = plot_cfr_vs_size_fine(dim_model_rates, save_path="plots/cfr_vs_size_fine_by_family.png")

In [37]:
# Polished barplot: Context-Following Rate by model_root (Base vs Instruct)
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

Path('plots').mkdir(exist_ok=True)

# Aggregate across dimensions for a cleaner comparison per model_root × type
bar_df = (
    dim_model_rates.groupby(['model_root', 'type'], as_index=False)['context_rate']
                   .mean()
)

# # Order models by their average (across types) context rate
# order = (
#     bar_df.groupby('model_root')['context_rate']
#           .mean()
#           .sort_values(ascending=False)
#           .index
# )

fig_height = 7
fig, ax = plt.subplots(figsize=(14, fig_height))

sns.barplot(
    data=dim_model_rates,
    y='model_root', x='context_rate',
    hue='type', 
    # order=order,
    palette='Set2', saturation=0.95, edgecolor='white', 
    # linewidth=2.0,
    ax=ax
)

# Axes and formatting
ax.set_xlim(0, 100)
# ax.axvline(50, color='black', linestyle='--', linewidth=1, alpha=0.4)
ax.set_title('Context-Following Rate by Model', fontsize=36, pad=14)
ax.set_xlabel('Context-Following Rate (%)', fontsize=30)
ax.set_ylabel('Model', fontsize=30)
# ax.grid(axis='x', linestyle='--', alpha=0.25)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.tick_params(axis='x', labelsize=30)
ax.tick_params(axis='y', labelsize=30)

# Make tick marks visible
ax.tick_params(axis='both', which='both', bottom=True, left=True, length=6, width=1.2, direction='out')

# Annotate bars with values (outside the bar)
# for p in ax.patches:
#     width = p.get_width()
#     y = p.get_y() + p.get_height() / 2
#     ax.text(min(width + 1.0, 99.5), y, f'{width:.1f}%', va='center', ha='left', fontsize=16, color='#333333')

# Legend outside
leg = ax.legend(title='Type', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
plt.setp(leg.get_texts(), fontsize=30)
plt.setp(leg.get_title(), fontsize=30)

plt.tight_layout()
plt.savefig('plots/context_following_rate_barplot_by_model_root_type.png', dpi=300, bbox_inches='tight')
plt.show()


In [38]:
category_summary_df

,category,dimension,total_examples,model,memory_pct,context_pct,other_pct,avg_yes_prob,avg_no_prob,avg_confidence,std_yes_prob,std_no_prob
0,authority,evidentiality,254,LLaMA3-8B-Instruct,57.480315,42.519685,0.0,-1.0,-1.0,0.0,0.0,0.0
1,authority,evidentiality,28,LLaMA3-8B,7.142857,92.857143,0.0,-1.0,-1.0,0.0,0.0,0.0
2,authority,evidentiality,70,LLaMA3-1B-Instruct,0.000000,100.000000,0.0,-1.0,-1.0,0.0,0.0,0.0
3,authority,evidentiality,38,LLaMA3-1B,31.578947,68.421053,0.0,-1.0,-1.0,0.0,0.0,0.0
4,authority,evidentiality,40,LLaMA3-3B-Instruct,5.000000,95.000000,0.0,-1.0,-1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
107,weak,epistemic_stance,10,LLaMA3-8B,60.000000,40.000000,0.0,-1.0,-1.0,0.0,0.0,0.0
108,weak,epistemic_stance,38,LLaMA3-1B-Instruct,10.526316,89.473684,0.0,-1.0,-1.0,0.0,0.0,0.0
109,weak,epistemic_stance,34,LLaMA3-1B,41.176471,58.823529,0.0,-1.0,-1.0,0.0,0.0,0.0
110,weak,epistemic_stance,52,LLaMA3-3B-Instruct,80.769231,19.230769,0.0,-1.0,-1.0,0.0,0.0,0.0


In [39]:
# Spearman correlation of context_pct rankings across models (by category)
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Choose value column
value_col = 'context_pct' if 'context_pct' in category_summary_df.columns else (
    'context_following_rate' if 'context_following_rate' in category_summary_df.columns else None
)
if value_col is None:
    raise KeyError(
        "Neither 'context_pct' nor 'context_following_rate' found in category_summary_df columns: "
        f"{list(category_summary_df.columns)}"
    )

required_columns = {'category', 'model', value_col}
missing_columns = required_columns.difference(category_summary_df.columns)
if missing_columns:
    raise KeyError(f"category_summary_df is missing required columns: {missing_columns}")

# Pivot to categories x models
pivot_df = category_summary_df.pivot_table(
    index='category',
    columns='model',
    values=value_col,
    aggfunc='mean'  # in case there are duplicate category-model rows
)

# Compute Spearman correlation across models (ranking across categories)
spearman_corr = pivot_df.corr(method='spearman')

# Plot heatmap
plt.figure(figsize=(14, 14))
sns.set_theme(style='whitegrid')
ax = sns.heatmap(
    spearman_corr,
    vmin=-1, vmax=1, cmap='coolwarm', center=0,
    annot=True, fmt='.2f', square=True,
    cbar_kws={'label': 'Spearman rho'}
)
ax.set_title('Spearman correlation of context_pct rankings across models (by category)')
ax.set_xlabel('Model')
ax.set_ylabel('Model')
plt.tight_layout()

# Save figure (supports running from repo root or analysis/)
plots_dir_candidates = [Path('plots'), Path('../plots')]
plots_dir = next((p for p in plots_dir_candidates if p.exists()), plots_dir_candidates[0])
plots_dir.mkdir(parents=True, exist_ok=True)
output_path = plots_dir / 'spearman_correlation_of_context_pct_rankings_by_model.png'
plt.savefig(output_path, dpi=200)
plt.show()

# Display the matrix as a quick table preview
spearman_corr.head()


model,LLaMA3-1B,LLaMA3-1B-Instruct,LLaMA3-3B,LLaMA3-3B-Instruct,LLaMA3-8B,LLaMA3-8B-Instruct
model,,,,,,
LLaMA3-1B,1.000000,0.559934,0.777193,0.324704,0.687719,0.150877
LLaMA3-1B-Instruct,0.559934,1.000000,0.344478,0.496049,0.613798,0.402100
LLaMA3-3B,0.777193,0.344478,1.000000,0.385257,0.692982,0.405263
LLaMA3-3B-Instruct,0.324704,0.496049,0.385257,1.000000,0.625713,0.667837
LLaMA3-8B,0.687719,0.613798,0.692982,0.625713,1.000000,0.608772


# Rank Analysis

## 15 Categories (filtering out missing data)

In [40]:
def add_star_legend(ax, alpha=0.05, star_color="black", star_edge="white", star_size=17):
    handles = [
        Line2D([0],[0], marker='*', linestyle='None', markersize=star_size,
               markerfacecolor=star_color, markeredgecolor=star_edge,
               label=f" q ≤ {alpha:g}"),
        Line2D([0],[0], marker='*', linestyle='None', markersize=star_size,
               markerfacecolor=star_color, markeredgecolor=star_edge,
               label=f" q ≤ {alpha/10:g}"),
    ]
    leg = ax.legend(handles=handles, # title="FDR significance",
                    loc="lower left", bbox_to_anchor=(0.09, 0.03), 
                    frameon=True, fontsize=16)
    return leg

# map category -> dimension
DIM_MAP = {
    # Form
    "explicit":"Form","not_at_issue":"Form","conditional":"Form",
    "counterfactual":"Form","imperative":"Form","interrogative":"Form",
    # Evidentiality
    "authority":"Evidentiality","belief_reports":"Evidentiality","hearsay":"Evidentiality",
    # Epistemic Stance
    "strong":"Epistemic Stance","weak":"Epistemic Stance",
    # Tone
    "formal":"Tone","informal":"Tone","poetic":"Tone",
    "social_media":"Tone","child_directed":"Tone","emotional_appeal":"Tone","sarcasm":"Tone",
}

def clean_category_name(cat):
    """Remove underscores and capitalize first letter"""
    return cat.replace('_', ' ').capitalize()

def _scores_df(category_summary_df):
    df = category_summary_df.copy()
    if "context_following_rate" not in df.columns:
        df["context_following_rate"] = (
            df["context_pct"] / (df["context_pct"] + df["memory_pct"])
        ) * 100
    return (df.groupby(["model","category"],as_index=False)["context_following_rate"]
              .mean().rename(columns={"context_following_rate":"score"}))

def _ranks_wide(scores_df):
    P = scores_df.pivot(index="model", columns="category", values="score").dropna(axis=1, how="any")
    R = P.rank(axis=1, ascending=False, method="average")
    return R, P.columns.tolist()

def _perm_p_two_sided(R, n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)
    M, K = R.shape
    expected = (K+1)/2.0
    obs = R.mean(axis=0)
    perm_means = np.empty((n_perm, K))
    for b in range(n_perm):
        PR = np.empty_like(R)
        for m in range(M):
            PR[m] = R[m, rng.permutation(K)]
        perm_means[b] = PR.mean(axis=0)
    dev_obs  = np.abs(obs - expected)
    dev_perm = np.abs(perm_means - expected)
    p_two = (np.sum(dev_perm >= dev_obs, axis=0) + 1) / (n_perm + 1)
    return obs, expected, p_two

def plot_rank_box_dim_stars_left(category_summary_df,
                                 alpha=0.05, n_perm=5000, seed=0,
                                 palette=None,
                                 star_color="black", star_edge="white",
                                 star_size=100,
                                 star_gap=0.035,
                                 left_margin=0.28,
                                 save_path="plots/rank_boxplot_sorted_dim_stars_left.png"):
    
    plt.rcParams.update({
        'font.size': 18,           # default text size
        'axes.titlesize': 22,      # title
        'axes.labelsize': 20,      # axis labels
        'xtick.labelsize': 18,     # x-axis tick labels
        'ytick.labelsize': 18,     # y-axis tick labels  
        'legend.fontsize': 18,     # legend text
        'legend.title_fontsize': 20  # legend title
    })

    # Better color palette
    if palette is None:
        palette = ["#1F77B4", "#FF7F0E", "#2CA02C", "#D62728"]  # blue, purple, orange, teal 


    # ranks across models
    scores = _scores_df(category_summary_df)
    R_wide, cats = _ranks_wide(scores)
    ranks_long = (R_wide.reset_index()
                  .melt(id_vars="model", var_name="category", value_name="rank"))
    ranks_long["dimension"] = ranks_long["category"].map(DIM_MAP).fillna("Other")
    
    # Clean category names for display
    ranks_long["category_display"] = ranks_long["category"].apply(clean_category_name)

    # order by mean rank
    order = (ranks_long.groupby("category")["rank"].mean()
             .sort_values().index.tolist())
    order_display = [clean_category_name(c) for c in order]
    K = len(order)

    # permutation test + BH/FDR
    obs, expected, p_two = _perm_p_two_sided(R_wide[order].to_numpy(), n_perm=n_perm, seed=seed)
    reject, qvals, _, _ = multipletests(p_two, alpha=alpha, method="fdr_bh")
    sig = pd.DataFrame({"category": order, "obs_mean_rank": obs,
                        "q_two_sided": qvals, "significant": reject})

    # plot
    dim_order = ["Form","Evidentiality","Epistemic Stance","Tone","Other"]
    pal = dict(zip(dim_order, list(palette)+["#999999"]))
    flierprops = dict(marker='o', markersize=3, markerfacecolor='#888',
                      markeredgecolor='none', alpha=0.45)

    fig, ax = plt.subplots(figsize=(14, max(6, 0.55*K)))
    
    sns.boxplot(
        data=ranks_long, x="rank", y="category_display",
        order=order_display, hue="dimension", dodge=False,
        palette=pal, linewidth=1.2, showfliers=True, flierprops=flierprops,
        ax=ax
    )
    ax.axvline((K+1)/2.0, ls="--", lw=2, color='gray', alpha=0.6)

    # draw star markers to the LEFT of y-labels
    yticklabels = ax.get_yticklabels()
    base_x = 0.032
    for i, cat in enumerate(order):
        q = sig.loc[sig["category"] == cat, "q_two_sided"].item()
        nstars = 0
        if q <= alpha/100: nstars = 3
        elif q <= alpha/10: nstars = 2
        elif q <= alpha: nstars = 1
        if nstars:
            for k in range(nstars):
                ax.scatter(base_x - k*star_gap, i, marker='*', s=star_size,
                           c=star_color, edgecolors=star_edge, linewidths=1.6,
                           transform=ax.get_yaxis_transform(),
                           clip_on=False, zorder=6)
            if i < len(yticklabels):
                yticklabels[i].set_fontweight("bold")

    # Move legend BELOW title (bbox y > 1 puts it above the plot area)
    handles, labels = ax.get_legend_handles_labels()
    dim_leg = ax.legend(handles, labels, 
                        loc="upper center",
                        bbox_to_anchor=(0.5, 1.09),
                        ncol=5, 
                        frameon=False,
                        fontsize=15)
    ax.add_artist(dim_leg)  # Keep dimension legend when adding star legend
    
    # Add star significance legend
    add_star_legend(ax, alpha=alpha, star_color=star_color, 
                    star_edge=star_edge, star_size=10)

    ax.set_title("Context-Following Ranks by Expressions of Belief Category", 
                 pad=40, fontsize=18)  # more padding for legend
    ax.set_xlabel("Rank (1 = most context-following, 17 = least context-following)")
    ax.set_ylabel("")  # Remove "category" label

    plt.tight_layout(rect=[left_margin, 0.02, 0.98, 0.92])

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    return sig

# Run it
sig_df = plot_rank_box_dim_stars_left(
    category_summary_df, alpha=0.05, n_perm=5000, seed=0,
    star_color="black", star_edge="none", star_size=80, star_gap=0.02,
    save_path="plots/rank_boxplot_sorted_dim_stars_left.png"
)
print(sig_df)

            category  obs_mean_rank  q_two_sided  significant
0     child_directed       4.000000     0.054389        False
1             formal       4.916667     0.147530        False
2          authority       5.583333     0.196736        False
3   emotional_appeal       5.833333     0.204337        False
4         imperative       6.000000     0.265827        False
5             strong       6.416667     0.317373        False
6        supposition       6.666667     0.330519        False
7           explicit       6.916667     0.374654        False
8             poetic       7.583333     0.551965        False
9           informal       8.250000     0.729654        False
10           sarcasm      10.333333     0.551965        False
11     interrogative      11.333333     0.330519        False
12      not_at_issue      12.500000     0.196736        False
13      social_media      12.666667     0.194328        False
14           hearsay      13.333333     0.121526        False
15      

## 17 Categories

In [41]:
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 1) keep ALL categories; rank per model on available categories
def _ranks_wide_allow_missing(scores_df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame R (models x categories) of ranks with NaNs where a
    model lacks that category. Ranks are computed within each model using
    only the categories that exist for that model (1 = best).
    """
    P = scores_df.pivot(index="model", columns="category", values="score")  # no dropna
    R = P.copy()
    for m, row in P.iterrows():
        s = row.dropna()
        r = s.rank(ascending=False, method="average")
        R.loc[m, r.index] = r
    return R  # rows=models, cols=ALL categories; NaN where missing

# 2) permutation test that tolerates NaNs (missing categories)
def _perm_p_two_sided_with_nan(R_df: pd.DataFrame, n_perm: int = 5000, seed: int = 0):
    """
    Two-sided permutation test on mean ranks per category with missing cells.
    For each model, we permute the non-NaN ranks across that model’s available
    categories; column means are computed with nanmean.
    The null expected mean for a category is the average of (K_m+1)/2 over
    the models m where that category is present, where K_m is the number of
    categories available for model m.
    """
    rng = np.random.default_rng(seed)
    R = R_df.to_numpy(copy=True)
    mask = ~np.isnan(R)           # M x K
    M, K = R.shape
    K_row = mask.sum(axis=1)      # K_m per model

    # observed and expected (per-category)
    obs = np.nanmean(R, axis=0)
    expected = np.empty(K)
    for j in range(K):
        mj = mask[:, j]
        expected[j] = np.mean((K_row[mj] + 1) / 2.0) if mj.any() else np.nan

    # permutations
    perm_means = np.empty((n_perm, K))
    for b in range(n_perm):
        perm_R = R.copy()
        for m in range(M):
            idx = np.where(mask[m])[0]
            if idx.size > 1:
                perm_R[m, idx] = perm_R[m, idx][rng.permutation(idx.size)]
        perm_means[b] = np.nanmean(perm_R, axis=0)

    # two-sided p-values (with finite-sample +1)/(B+1)
    dev_obs = np.abs(obs - expected)
    dev_perm = np.abs(perm_means - expected)
    p_two = (np.sum(dev_perm >= dev_obs, axis=0) + 1) / (n_perm + 1)

    return obs, expected, p_two, perm_means

# 3) plug the new functions into your plotting routine
def plot_rank_box_dim_stars_left(category_summary_df,
                                 alpha=0.05, n_perm=5000, seed=0,
                                 palette=None,
                                 star_color="black", star_edge="white",
                                 star_size=100, star_gap=0.035,
                                 left_margin=0.28,
                                 save_path="plots/rank_boxplot_sorted_dim_stars_left.png"):
    if palette is None:
        palette = ["#1F77B4", "#FF7F0E", "#2CA02C", "#D62728"]

    # scores per (model, category)
    scores = _scores_df(category_summary_df)

    # ranks (keep ALL categories)
    R_wide = _ranks_wide_allow_missing(scores)         # DataFrame with NaNs allowed
    ranks_long = (R_wide.reset_index()
                  .melt(id_vars="model", var_name="category", value_name="rank"))
    ranks_long = ranks_long.dropna(subset=["rank"])    # seaborn ignores NaNs anyway
    ranks_long["dimension"] = ranks_long["category"].map(DIM_MAP).fillna("Other")
    ranks_long["category_display"] = ranks_long["category"].apply(clean_category_name)

    # order by mean rank (lower = better)
    order = (ranks_long.groupby("category")["rank"].mean()
             .sort_values().index.tolist())
    order_display = [clean_category_name(c) for c in order]

    # permutation test with NaNs
    obs, expected, p_two, _ = _perm_p_two_sided_with_nan(R_wide[order], n_perm=n_perm, seed=seed)
    # FDR (handle NaNs safely)
    valid = np.isfinite(p_two)
    qvals = np.full_like(p_two, np.nan, dtype=float)
    reject = np.zeros_like(p_two, dtype=bool)
    if valid.any():
        rej_v, q_v, _, _ = multipletests(p_two[valid], alpha=alpha, method="fdr_bh")
        qvals[valid] = q_v
        reject[valid] = rej_v
    sig = pd.DataFrame({"category": order, "obs_mean_rank": obs,
                        "expected_mean_rank": expected, "p_two_sided": p_two,
                        "q_two_sided": qvals, "significant": reject})

    # plot
    dim_order = ["Form","Evidentiality","Epistemic Stance","Tone","Other"]
    pal = dict(zip(dim_order, list(palette)+["#999999"]))
    flierprops = dict(marker='o', markersize=3, markerfacecolor='#888',
                      markeredgecolor='none', alpha=0.45)

    fig, ax = plt.subplots(figsize=(14, max(6, 0.55*len(order))))
    sns.boxplot(data=ranks_long, x="rank", y="category_display",
                order=order_display, hue="dimension", dodge=False,
                palette=pal, linewidth=1.2, showfliers=True, flierprops=flierprops, ax=ax)

    # reference line at the overall expected mean (averaged across categories)
    exp_global = float(np.nanmean(expected))
    ax.axvline(exp_global, ls="--", lw=2, color='gray', alpha=0.6)

    # left-side stars
    yticklabels = ax.get_yticklabels()
    base_x = 0.032
    for i, cat in enumerate(order):
        q = sig.loc[sig["category"] == cat, "q_two_sided"].item()
        nstars = 0
        if np.isfinite(q):
            if q <= alpha/100: nstars = 3
            elif q <= alpha/10: nstars = 2
            elif q <= alpha: nstars = 1
        for k in range(nstars):
            ax.scatter(base_x - k*star_gap, i, marker='*', s=star_size,
                       c=star_color, edgecolors=star_edge, linewidths=1.6,
                       transform=ax.get_yaxis_transform(), clip_on=False, zorder=6)
        if nstars and i < len(yticklabels):
            yticklabels[i].set_fontweight("bold")

    # legends
    handles, labels = ax.get_legend_handles_labels()
    dim_leg = ax.legend(handles, labels, loc="upper center",
                        bbox_to_anchor=(0.5, 1.09), ncol=5, frameon=False, fontsize=15)
    ax.add_artist(dim_leg)
    add_star_legend(ax, alpha=alpha, star_color=star_color, star_edge=star_edge, star_size=10)

    ax.set_title("Context-Following Ranks by Expressions of Belief Category",
                 pad=40, fontsize=20)
    ax.set_xlabel("Rank (1 = most context-following, 17 = least context-following)")
    ax.set_ylabel("")
    plt.tight_layout(rect=[left_margin, 0.02, 0.98, 0.92])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    return sig


sig_df = plot_rank_box_dim_stars_left(
    category_summary_df, alpha=0.05, n_perm=5000, seed=0,
    star_color="black", star_edge="none", star_size=80, star_gap=0.02,
    save_path="plots/rank_boxplot_sorted_dim_stars_left_new.png"
)
print(len(sig_df["category"].unique()))  # should be 17
print(sig_df)

19
                category  obs_mean_rank  expected_mean_rank  p_two_sided  \
0         child_directed       4.000000            9.833333     0.006199   
1                 formal       4.916667            9.833333     0.022196   
2              authority       5.750000            9.833333     0.058788   
3       emotional_appeal       6.000000            9.833333     0.077584   
4             imperative       6.000000            9.833333     0.081384   
5            supposition       6.666667            9.833333     0.154369   
6                 strong       6.833333            9.833333     0.167367   
7               explicit       7.083333            9.833333     0.216357   
8                 poetic       7.916667            9.833333     0.393321   
9               informal       8.416667            9.833333     0.520096   
10               sarcasm      10.666667            9.833333     0.724255   
11         interrogative      11.333333            9.833333     0.509098   
12  mater

# Detailed Inisghts for Base and Instruct CFR along dimensions

In [42]:

import math
from pathlib import Path

Path("plots").mkdir(exist_ok=True)

# ------------------------------
# helpers
# ------------------------------
def _detect_cols(df, model_col=None, type_col=None, dim_col=None, val_col=None):
    cols = {c.lower(): c for c in df.columns}
    def pick(candidates, given):
        if given and given in df.columns: return given
        for c in candidates:
            if c in df.columns: return c
            if c.lower() in cols: return cols[c.lower()]
        raise KeyError(f"Need one of {candidates} in columns={list(df.columns)}")
    model = pick(["model_root","model","model_name"], model_col)
    typ   = pick(["type","model_type","train_type","variant"], type_col)
    dim   = pick(["dimension","dim","category","assertion_dimension"], dim_col)
    val   = pick(["context_rate","context_following_rate","cfr","score"], val_col)
    return model, typ, dim, val

def _sanitize(df, dim_col):
    # string-clean dimension labels; keep whatever you have
    out = df.copy()
    out[dim_col] = (out[dim_col].astype(str)
                    .str.strip()
                    .str.replace(r"\s+", " ", regex=True))
    return out

# ------------------------------
# 1) HEATMAPS (absolute + Δ vs model mean)
# ------------------------------
def plot_dimension_heatmaps_generic(dim_model_rates: pd.DataFrame,
                                    model_col=None, type_col=None, dim_col=None, val_col=None,
                                    annotate=True,
                                    save_prefix="plots/dimension_heatmaps_generic"):
    mcol, tcol, dcol, vcol = _detect_cols(dim_model_rates, model_col, type_col, dim_col, val_col)
    df = _sanitize(dim_model_rates[[mcol, tcol, dcol, vcol]].dropna(), dcol)

    # assure numeric
    df[vcol] = pd.to_numeric(df[vcol], errors="coerce")
    df = df.dropna(subset=[vcol])
    if df.empty:
        raise ValueError("After coercing to numeric, no data left. Check your value column.")

    # consistent dimension order = natural sorted
    dim_order = sorted(df[dcol].unique(), key=lambda s: s.lower())

    for typ in sorted(df[tcol].unique()):
        sub = df[df[tcol] == typ]
        if sub.empty: 
            continue

        piv_abs = sub.pivot_table(index=mcol, columns=dcol, values=vcol, aggfunc="mean")
        piv_abs = piv_abs.reindex(columns=dim_order)
        # order models by row mean
        row_order = piv_abs.mean(axis=1).sort_values(ascending=False).index
        piv_abs = piv_abs.reindex(index=row_order)

        # absolute heatmap
        plt.figure(figsize=(1.8 + 2.0*len(dim_order), 0.6 + 0.55*len(piv_abs)))
        ax = sns.heatmap(piv_abs, cmap="viridis", vmin=0, vmax=100,
                         annot=annotate, fmt=".0f",
                         cbar_kws={"label": "CFR (%)"})
        ax.set_title(f"Dimension CFR by Model — {typ}")
        ax.set_xlabel("Dimension"); ax.set_ylabel("Model")
        plt.tight_layout()
        plt.savefig(f"{save_prefix}_{str(typ).lower()}_absolute.png", dpi=300, bbox_inches="tight")
        plt.close()

        # delta heatmap (subtract each model's average)
        piv_delta = piv_abs.sub(piv_abs.mean(axis=1), axis=0)
        vmax = np.nanmax(np.abs(piv_delta.values))
        vmax = 1.0 if not np.isfinite(vmax) or vmax == 0 else vmax

        plt.figure(figsize=(1.8 + 2.0*len(dim_order), 0.6 + 0.55*len(piv_delta)))
        ax = sns.heatmap(piv_delta, cmap="coolwarm", center=0, vmin=-vmax, vmax=vmax,
                         annot=annotate, fmt="+.1f",
                         cbar_kws={"label": "Δ CFR vs model mean (pp)"})
        ax.set_title(f"Dimension CFR Δ vs Model Mean — {typ}")
        ax.set_xlabel("Dimension"); ax.set_ylabel("Model")
        plt.tight_layout()
        plt.savefig(f"{save_prefix}_{str(typ).lower()}_delta.png", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

# ------------------------------
# 2) SMALL-MULTIPLES (bars per model × dimension)
# ------------------------------
def plot_per_model_dimension_bars_generic(dim_model_rates: pd.DataFrame,
                                          model_col=None, type_col=None, dim_col=None, val_col=None,
                                          ncols=3,
                                          save_path="plots/per_model_dimension_bars_generic.png"):
    mcol, tcol, dcol, vcol = _detect_cols(dim_model_rates, model_col, type_col, dim_col, val_col)
    df = _sanitize(dim_model_rates[[mcol, tcol, dcol, vcol]].dropna(), dcol)
    df[vcol] = pd.to_numeric(df[vcol], errors="coerce")
    df = df.dropna(subset=[vcol])

    # aggregate to unique (model, type, dimension)
    df = df.groupby([mcol, tcol, dcol], as_index=False)[vcol].mean()

    # ordered categorical on the column (no reindex-on-duplicates)
    dim_order = sorted(df[dcol].unique(), key=lambda s: s.lower())
    df[dcol] = pd.Categorical(df[dcol], categories=dim_order, ordered=True)

    models = sorted(df[mcol].unique(), key=lambda s: str(s))
    nrows = math.ceil(len(models) / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6.5*ncols, 4.2*nrows))
    axes = np.array(axes).reshape(-1)

    drew_any = False
    for ax, model in zip(axes, models):
        m = df[df[mcol] == model].sort_values(dcol)
        if m.empty:
            ax.axis("off"); continue
        sns.barplot(data=m, x=dcol, y=vcol, hue=tcol, ax=ax,
                    palette="Set2", edgecolor="white", saturation=0.95, ci=None)
        ax.set_title(str(model), fontsize=12)
        ax.set_xlabel(""); ax.set_ylabel("CFR (%)")
        ax.set_ylim(0, 100); ax.tick_params(axis="x", rotation=30)
        if ax.legend_: ax.legend_.remove()
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        drew_any = True

    for ax in axes[len(models):]:
        ax.axis("off")

    if drew_any:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, title="Type", loc="upper center", ncol=2, frameon=False)
        fig.suptitle("Context-Following Rate × Type × Dimension (per model)", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

# ------------------------------
# Run with your DataFrame
# ------------------------------
plot_dimension_heatmaps_generic(dim_model_rates, annotate=True,
                                save_prefix="plots/dimension_heatmaps_generic")

plot_per_model_dimension_bars_generic(dim_model_rates, ncols=3,
                                      save_path="plots/per_model_dimension_bars_generic.png")
